# Schwarzfahrer im Schweizer ÖV

Gruppe 05: Dominik Erbsland, Joel Liechti. ZHAW, CAS Information Engineering, Scripting.

**Fragestellung:** Auf welchen Linien, an welchen Haltestellen und zu welchen Zeiten werden im Schweizer ÖV besonders viele Schwarzfahrer festgestellt?

**Daten**
- Kontrollen mit Ergebnis "Schwarzfahrt" (Excel-Export `ASP_202509.XLSX`, eigener Datensatz)
- Haltestellen mit Koordinaten (SBB Atlas, opendata.swiss)

**Ablauf:** 1. Daten einlesen, 2. Daten ansehen, 3. Bereinigen, 4. Haltestellen zuordnen, 5. Statistiken, 6. Darstellung und Karte. Jeder Schritt steht mit Code und Begründung in diesem Notebook.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RAW = Path.cwd().parent / "data" / "raw"  # Notebook liegt in notebooks/

## 1. Daten einlesen

### 1.1 Haltestellen

Die CSV-Datei hat `;` als Trennzeichen und ist UTF-8 mit BOM. Die Koordinaten stehen als Text in einer Spalte (`"46.55, 6.61"`).

In [ ]:
stops_raw = pd.read_csv(RAW / "haltestelle_haltekante.csv", sep=";", encoding="utf-8-sig", low_memory=False)
print(stops_raw.shape)
stops_raw[["Haltestelle SLOID", "geopos_haltestelle", "offizielle Haltestellen Bezeichnung", "meansOfTransport"]].head()

In [ ]:
# Koordinaten in lat und lon aufteilen und nur die benötigten Spalten behalten
coords = stops_raw["geopos_haltestelle"].str.split(",", expand=True).astype(float)

stops = pd.DataFrame({
    "sloid": stops_raw["Haltestelle SLOID"],
    "name": stops_raw["offizielle Haltestellen Bezeichnung"].str.strip(),
    "lat": coords[0],
    "lon": coords[1],
    "canton": stops_raw["cantonAbbreviation"],
    "transport": stops_raw["meansOfTransport"],
    "operator": stops_raw["businessOrganisationDescriptionDe"],
})

# Zeilen ohne Koordinaten entfernen, eine Zeile pro Haltestelle behalten
stops = stops.dropna(subset=["lat", "lon"]).drop_duplicates("sloid").reset_index(drop=True)
print(stops.shape, "| Namen eindeutig:", stops["name"].is_unique)
stops.head()

### 1.2 Kontrolldaten

Jede Zeile ist eine Kontrolle mit Ergebnis "Schwarzfahrt": Verkehrsunternehmen (`DRIVING_COMPANY`), Kontrolldatum und -zeit, Linie, Start- und Zielhaltestelle, Geschlecht, Geburtsdatum und Grund.

In [ ]:
ASP_FILE = RAW / "ASP_202509.XLSX"
if not ASP_FILE.exists():
    raise FileNotFoundError(
        f"{ASP_FILE.name} fehlt. Die Datei ist nicht im Repository (enthält Geburtsdaten). "
        f"Bitte lokal in {RAW} ablegen, siehe README."
    )

controls_raw = pd.read_excel(ASP_FILE)
print(controls_raw.shape)
controls_raw.head()

## 2. Daten ansehen

Bevor wir bereinigen, prüfen wir Qualität und Umfang.

In [ ]:
print("Fehlende Werte:")
print(controls_raw.isna().sum()[lambda s: s > 0])
print("\nDuplikate:", controls_raw.duplicated().sum())
print("Start gleich Ziel:", (controls_raw["STATION_FROM"] == controls_raw["STATION_TO"]).sum())

In [ ]:
# Zeitraum: Der Export ist vom September 2025, enthält aber auch ältere Kontrollen (Nachimport)
controls_raw["CONTROL_DATE"].str[:7].value_counts().sort_index().tail(8)

In [ ]:
# Verkehrsunternehmen: es sind nicht nur PostAuto-Fahrten (801)
print("Unternehmen:", controls_raw["DRIVING_COMPANY"].nunique())
controls_raw["DRIVING_COMPANY"].value_counts().head(8)

In [ ]:
# Gründe: Freitext in mehreren Sprachen
print("Verschiedene Gründe:", controls_raw["REASON"].nunique())
controls_raw["REASON"].value_counts().head(12)

## 3. Bereinigen

- **Duplikate** entfernen.
- **Datum und Zeit** zu einem `datetime` zusammenfügen, daraus Wochentag und Stunde ableiten.
- **Geburtsdatum:** Wir behalten nur Jahrgang und Altersgruppe und löschen das volle Geburtsdatum. Zusammen mit Ort und Zeit könnte es Personen identifizierbar machen.
- **Gründe:** Rund 540 verschiedene Freitexte (Deutsch, Französisch, Italienisch) fassen wir in wenige Kategorien zusammen.
- **Start gleich Ziel:** markieren, nicht löschen.

In [ ]:
controls = controls_raw.drop_duplicates().copy()
print("Zeilen vor:", len(controls_raw), "| nach:", len(controls))

controls["datetime"] = pd.to_datetime(controls["CONTROL_DATE"] + " " + controls["CONTROL_TIME"])
controls["weekday"] = controls["datetime"].dt.day_of_week  # 0 = Montag
controls["hour"] = controls["datetime"].dt.hour

In [ ]:
# Jahrgang (nur das Jahr, nicht das volle Geburtsdatum) und Alter zum Kontrollzeitpunkt
birthdate = pd.to_datetime(controls["BIRTHDATE"])
controls["birth_year"] = birthdate.dt.year.astype("Int64")  # Int64 erlaubt ganze Zahlen mit fehlenden Werten
age = (controls["datetime"] - birthdate).dt.days // 365
controls["age_group"] = pd.cut(age, [0, 17, 25, 35, 50, 65, 120],
                               labels=["<18", "18-25", "26-35", "36-50", "51-65", ">65"])
controls["age_group"].value_counts(dropna=False).sort_index()

In [ ]:
# Gründe in Kategorien (Schlüsselwörter in DE, FR und IT). Die Reihenfolge zählt: die erste passende Kategorie gewinnt.
categories = {
    "E-Ticket / Check-In": r"e-ticket|check-in|checkin|e-billet",
    "Abo abgelaufen": r"abo|abonnement|abgelaufen|expir",
    "Zone / Strecke fehlt": r"zone|haltestelle|arr[eê]t|strecke|trajet|fermata",
    "Falsche Klasse / Kundengruppe": r"klasse|kundengruppe|classe|ermäss|réduit|reduit",
    "Kein gültiges Billett": r"ohne g|kein|sans titre|senza titolo|billett|fahrausweis|titre de transport|biglietto",
}

def categorize(text):
    text = str(text).lower()
    for name, pattern in categories.items():
        if re.search(pattern, text):
            return name
    return "Andere"

controls["reason_cat"] = controls["REASON"].map(categorize)
controls["reason_cat"].value_counts()

In [ ]:
# Kontrolle: was landet in "Andere"?
controls.loc[controls["reason_cat"] == "Andere", "REASON"].value_counts().head(10)

**Unternehmensnamen:** `DRIVING_COMPANY` enthält nur Nummern. Die Haltestellendaten kennen zu jeder Nummer den Namen (`businessOrganisationNumber`). Doppelte Kürzel wie "BLS-bls" kürzen wir auf "BLS". Nummern ohne Eintrag heissen "Unbekannt". Die Nummer 490 ordnen wir vermutungsweise dem ZVV zu ("ZVV (?)"), weil fast alle Fälle im Raum Zürich liegen. Das muss noch bestätigt werden.

In [ ]:
company_names = (stops_raw[["businessOrganisationNumber", "businessOrganisationAbbreviationDe"]]
                .dropna().drop_duplicates("businessOrganisationNumber")
                .set_index("businessOrganisationNumber")["businessOrganisationAbbreviationDe"])

# Doppelte Kürzel bereinigen: "BLS-bls" wird "BLS", "SOB-sob" wird "SOB"
company_names = company_names.str.replace(r"^(.+)-\1$", r"\1", case=False, regex=True)

# Von Hand ergänzt: 490 ist vermutet aufgrund der Haltestellen (Zürich), nicht bestätigt
manual_names = {490: "ZVV (?)", 801: "PostAuto"}  # 801 = PostAuto AG (Kürzel PAG ist kaum bekannt)

controls["company"] = (controls["DRIVING_COMPANY"].map(manual_names)
                       .fillna(controls["DRIVING_COMPANY"].map(company_names))
                       .fillna("Unbekannt"))
print(f"Unbekannt: {(controls['company'] == 'Unbekannt').mean():.1%} der Fälle")
controls["company"].value_counts().head(10)

In [ ]:
controls["same_from_to"] = controls["STATION_FROM"] == controls["STATION_TO"]

# Nicht mehr benötigte oder sensible Spalten entfernen
controls = controls.drop(columns=["BIRTHDATE", "CONTROL_DATE", "CONTROL_TIME", "IMPORT_DATE", "ACTION"])
controls.head()

## 4. Haltestellen zuordnen (Matching)

Wir verbinden `STATION_FROM` und `STATION_TO` über den Haltestellennamen mit den Haltestellendaten und erhalten so Koordinaten für die Karte.

In [ ]:
stops_by_name = stops.set_index("name")

for side, col in [("from", "STATION_FROM"), ("to", "STATION_TO")]:
    name = controls[col].str.strip()
    controls[f"lat_{side}"] = name.map(stops_by_name["lat"])
    controls[f"lon_{side}"] = name.map(stops_by_name["lon"])
    print(f"Treffer {side}: {controls[f'lat_{side}'].notna().mean():.1%}")

In [ ]:
# Was lässt sich nicht zuordnen? (Auslandstationen, abweichende Schreibweisen)
controls.loc[controls["lat_from"].isna(), "STATION_FROM"].value_counts().head(15)

## 5. Statistiken

### 5.1 Welche Jahrgänge werden am häufigsten erfasst?

In [ ]:
by_year = controls["birth_year"].value_counts().sort_index()
by_year = by_year[by_year.index >= 1930]  # unplausible Jahrgänge ausblenden

ax = by_year.plot(kind="bar", figsize=(12, 4), width=0.9)
ax.set_xlabel("Jahrgang")
ax.set_ylabel("Anzahl Fälle")
ax.set_title("Schwarzfahrten nach Jahrgang")
ax.set_xticks(range(0, len(by_year), 5))
ax.set_xticklabels(by_year.index[::5], rotation=0)
plt.show()

by_year.sort_values(ascending=False).head(10)

**Achtung:** Absolute Zahlen zeigen nur, welche Jahrgänge am häufigsten *erwischt* wurden. Sie sagen nicht, wer am häufigsten schwarzfährt, denn es gibt je nach Jahrgang unterschiedlich viele ÖV-Nutzer und unterschiedlich viele Kontrollen. Für einen fairen Vergleich bräuchten wir die Fahrgastzahlen pro Jahrgang.

### 5.2 Fälle pro Verkehrsunternehmen

In [ ]:
controls["company"].value_counts().head(10).plot(kind="barh", figsize=(8, 4)).invert_yaxis()
plt.xlabel("Anzahl Fälle")
plt.show()

### 5.3 Frauen und Männer
Der Code in `GENDER` ist nicht dokumentiert (1, 2 und 3). Wir nehmen die Norm ISO 5218 an (1 = männlich, 2 = weiblich, 3 = andere/unbekannt). Das ist **nicht bestätigt** und muss mit Joel bzw. PostAuto geklärt werden, bevor wir Aussagen daraus ableiten.

In [ ]:
gender_labels = {1: "männlich (?)", 2: "weiblich (?)", 3: "andere (?)"}
controls["gender"] = controls["GENDER"].map(gender_labels).fillna("unbekannt")

gender_counts = controls["gender"].value_counts()
print((gender_counts / len(controls)).map("{:.1%}".format))
gender_counts.plot(kind="bar", figsize=(6, 3), rot=0)
plt.ylabel("Anzahl Fälle")
plt.show()

In [ ]:
# Anteil pro Altersgruppe: unterscheidet sich das Verhältnis je nach Alter?
share = pd.crosstab(controls["age_group"], controls["gender"], normalize="index")
share.plot(kind="bar", stacked=True, figsize=(8, 4), rot=0)
plt.ylabel("Anteil")
plt.title("Geschlecht nach Altersgruppe")
plt.show()

### 5.4 Altersgruppen

In [ ]:
controls["age_group"].value_counts().sort_index().plot(kind="bar", figsize=(7, 3), rot=0)
plt.ylabel("Anzahl Fälle")
plt.show()

### 5.5 Wochentag und Tageszeit

In [ ]:
days = ["Mo", "Di", "Mi", "Do", "Fr", "Sa", "So"]
controls["weekday"].value_counts().sort_index().rename(dict(enumerate(days))).plot(kind="bar", figsize=(7, 3), rot=0)
plt.ylabel("Anzahl Fälle")
plt.title("Fälle pro Wochentag")
plt.show()

controls["hour"].value_counts().sort_index().plot(kind="bar", figsize=(10, 3), rot=0)
plt.xlabel("Stunde")
plt.ylabel("Anzahl Fälle")
plt.title("Fälle pro Tageszeit")
plt.show()

In [ ]:
# Wochentag und Stunde kombiniert (wann lohnen sich Kontrollen am meisten?)
heat_time = pd.crosstab(controls["weekday"], controls["hour"]).rename(index=dict(enumerate(days)))
fig, ax = plt.subplots(figsize=(12, 3.5))
im = ax.imshow(heat_time, aspect="auto", cmap="YlOrRd")
ax.set_yticks(range(len(heat_time.index)), heat_time.index)
ax.set_xticks(range(len(heat_time.columns)), heat_time.columns)
ax.set_xlabel("Stunde")
fig.colorbar(im, label="Anzahl Fälle")
plt.show()

**Achtung:** Auch hier zeigen die Zahlen, wann kontrolliert wurde und Fälle gefunden wurden. Wenn zu bestimmten Zeiten mehr kontrolliert wird, erscheinen dort auch mehr Fälle.

### 5.6 Gründe

In [ ]:
controls["reason_cat"].value_counts().plot(kind="barh", figsize=(8, 3)).invert_yaxis()
plt.xlabel("Anzahl Fälle")
plt.show()

# Gründe nach Verkehrsunternehmen (nur die 5 grössten)
top5 = controls["company"].value_counts().head(5).index
pd.crosstab(controls.loc[controls["company"].isin(top5), "company"], controls["reason_cat"], normalize="index").round(2)

### 5.7 Häufigste Haltestellen und Strecken

In [ ]:
print("Starthaltestellen")
display(controls["STATION_FROM"].value_counts().head(10))

controls["route"] = controls["STATION_FROM"].str.strip() + " → " + controls["STATION_TO"].str.strip()
print("Strecken")
display(controls.loc[~controls["same_from_to"], "route"].value_counts().head(10))

## 6. Darstellung

### 6.1 Karte (Heatmap)

In [ ]:
import folium
from folium.plugins import HeatMap

# Anzahl Fälle pro Starthaltestelle
heat = (controls.dropna(subset=["lat_from", "lon_from"])
        .groupby(["lat_from", "lon_from"]).size().reset_index(name="anzahl"))

m = folium.Map(location=[46.8, 8.2], zoom_start=8, tiles="OpenStreetMap")
HeatMap(heat[["lat_from", "lon_from", "anzahl"]].values.tolist(), radius=12).add_to(m)
m